# Lab 4 — Solving $Ax = b$: The Four Cases
**Linear Algebra · UATX**

Solvability of $Ax = b$ is governed by two independent questions:
1. Is $b \in \mathrm{im}(A)$? (existence)
2. Is $\ker(A) = \{0\}$? (uniqueness)

This gives a $2 \times 2$ table:

| | $\ker(A) = \{0\}$ | $\ker(A) \neq \{0\}$ |
|---|---|---|
| $b \in \mathrm{im}(A)$ | **Unique solution** | **Infinitely many** ($x_0 + \ker A$) |
| $b \notin \mathrm{im}(A)$ | **No solution** | **No solution** |

**Tasks**
1. Construct matrices and right-hand sides realising each of the four cases.
2. Extract a particular solution $x_0$ and parameterise the full solution set.
3. Compute and verify the least-norm solution.
4. Visualise solution sets in $\mathbb{R}^2$ and $\mathbb{R}^3$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.linalg import null_space

np.random.seed(42)

# ----------------------------------------------------------------
# Utility: column-reduction algorithm from Lab 3
# ----------------------------------------------------------------
def column_reduce(A, tol=1e-10):
    """
    Reduce [A ; I_n] by column operations.
    Returns (reduced, pivot_cols).
    pivot_cols  : indices of linearly independent columns of A
    reduced[:m] : echelon form of A (pivot columns = standard basis vectors)
    reduced[m:] : kernel basis in non-pivot columns; inv(A) if A is square full-rank
    """
    m, n = A.shape
    AI = np.vstack([A.astype(float), np.eye(n)])
    used_rows, pivot_positions = set(), []
    for col in range(n):
        for r in range(m):
            if r not in used_rows and abs(AI[r, col]) > tol:
                used_rows.add(r); pivot_positions.append((r, col))
                AI[:, col] /= AI[r, col]
                for other in range(n):
                    if other != col:
                        AI[:, other] -= AI[r, other] * AI[:, col]
                break
    pivot_positions.sort()
    pivot_cols = [c for r,c in pivot_positions]
    free_cols  = [j for j in range(n) if j not in pivot_cols]
    reduced = AI[:, pivot_cols + free_cols]
    return reduced, pivot_cols

## §1  The Four Cases

In [ ]:
# Case 1: Unique solution  (ker(A)={0},  b in im(A))
A1 = np.array([[2., 1.], [1., 3.]])
b1 = np.array([5., 7.])
x1 = np.linalg.solve(A1, b1)
print('=== Case 1: Unique solution ===')
print('A ='); print(A1)
print('b =', b1)
print('Solution x =', x1)
print('Residual ||Ax-b|| =', np.linalg.norm(A1 @ x1 - b1))
print('dim ker(A) =', A1.shape[1] - np.linalg.matrix_rank(A1))

In [ ]:
# Case 2: Infinitely many solutions  (ker(A)!={0},  b in im(A))
# A has rank 2 in R^{3x5}: kernel is 3-dimensional
A2 = np.array([[1., 0., 1., 2., -1.],
               [0., 1., 1., -1., 2.],
               [1., 1., 2., 1., 1.]], dtype=float)
b2 = np.array([1., 2., 3.])   # b = A[:,0] + 2*A[:,1], so in im(A)

print('=== Case 2: Infinitely many solutions ===')
print('rank(A) =', np.linalg.matrix_rank(A2))
print('dim ker(A) =', A2.shape[1] - np.linalg.matrix_rank(A2))

# Particular solution via lstsq
x0, _, _, _ = np.linalg.lstsq(A2, b2, rcond=None)
print('Particular solution x0 =', np.round(x0, 4))
print('A x0 = b?', np.allclose(A2 @ x0, b2))

# Kernel basis
K = null_space(A2)
print('Kernel basis (columns):', K.shape[1], 'vectors')

# Verify any point in the coset is a solution
alpha = np.random.randn(K.shape[1])
x_general = x0 + K @ alpha
print('A(x0 + K alpha) = b?', np.allclose(A2 @ x_general, b2))

In [ ]:
# Case 3: No solution  (b not in im(A))
A3 = np.array([[1., 0.], [0., 1.], [1., 1.]], dtype=float)   # rank 2, im = span{e1,e2} in R^3... wait
# Make a proper no-solution case: overdetermined with inconsistent b
A3 = np.array([[1., 2.], [3., 6.], [2., 4.]], dtype=float)   # rank 1
b3_ok  = np.array([1., 3., 2.])   # b in im(A): b = 1 * [1,3,2]^T
b3_bad = np.array([1., 4., 2.])   # b not in im(A)

print('=== Case 3: No solution ===')
print('rank(A) =', np.linalg.matrix_rank(A3))
print('rank([A|b_ok])  =', np.linalg.matrix_rank(np.hstack([A3, b3_ok.reshape(-1,1)])))
print('rank([A|b_bad]) =', np.linalg.matrix_rank(np.hstack([A3, b3_bad.reshape(-1,1)])))
print('b_bad in im(A):', np.linalg.matrix_rank(np.hstack([A3, b3_bad.reshape(-1,1)])) == np.linalg.matrix_rank(A3))

## §2  Parameterising the Full Solution Set

If $Ax = b$ has a solution $x_0$, then the **general solution** is $x_0 + \ker(A)$: all vectors $x_0 + k$ with $k \in \ker(A)$.

We extract $x_0$ using `lstsq` (which returns the minimum-norm particular solution) and $\ker(A)$ using `scipy.linalg.null_space`.

In [ ]:
# 3x5 matrix, rank 2, consistent b
A = np.array([[1., 2., 3., 4., 5.],
              [2., 4., 6., 8., 10.],
              [1., 1., 1., 1., 1.]], dtype=float)
b = np.array([15., 30., 5.])   # b = [15, 30, 5] = 5*[1,2,1] + ... in im(A)? Let's check
print('rank(A)       =', np.linalg.matrix_rank(A))
print('rank([A|b])   =', np.linalg.matrix_rank(np.hstack([A, b.reshape(-1,1)])))

# Particular solution (min-norm)
x0, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
K = null_space(A)
print('\nParticular solution x0 =', np.round(x0, 4))
print('Kernel dimension:', K.shape[1])

# Verify 100 random points in the solution set
for _ in range(100):
    alpha = np.random.randn(K.shape[1])
    x = x0 + K @ alpha
    assert np.linalg.norm(A @ x - b) < 1e-8, 'Failed!'
print('All 100 random points in x0 + ker(A) satisfy Ax=b: PASS')

## §3  The Least-Norm Solution

When $\ker(A) \neq \{0\}$, there are infinitely many solutions. The **least-norm solution** $x^* = \arg\min_{Ax=b} \|x\|$ is the unique solution orthogonal to $\ker(A)$, i.e., $x^* \in \ker(A)^\perp$.

`np.linalg.lstsq` always returns $x^*$.

In [ ]:
def least_norm_solution(A, b):
    x_star, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
    return x_star

# Using A2, b2 from §1
x_star = least_norm_solution(A2, b2)
K2 = null_space(A2)

print('Least-norm solution x* =', np.round(x_star, 4))
print('||x*|| =', np.round(np.linalg.norm(x_star), 4))
print('A x* = b:', np.allclose(A2 @ x_star, b2))

# Verify x* is orthogonal to all kernel vectors
for i in range(K2.shape[1]):
    inner = np.dot(x_star, K2[:, i])
    print(f'<x*, k{i+1}> = {inner:.2e}  (should be ~0)')

In [ ]:
# Compare norms: x* has smaller norm than any other solution
n_trials = 500
norms = []
for _ in range(n_trials):
    alpha = np.random.randn(K2.shape[1]) * 3
    x = x_star + K2 @ alpha
    norms.append(np.linalg.norm(x))

plt.figure(figsize=(6, 3))
plt.hist(norms, bins=40, alpha=0.7, label='||x0 + K alpha||')
plt.axvline(np.linalg.norm(x_star), color='red', lw=2, label=f'||x*|| = {np.linalg.norm(x_star):.3f}')
plt.xlabel('Norm'); plt.ylabel('Count')
plt.title('Least-norm solution is the minimum-norm element of $x^* + \\ker(A)$')
plt.legend(); plt.tight_layout(); plt.show()

## §4  Visualisation

### $\mathbb{R}^2$: solution set of $x_1 + 2x_2 = 3$
The solution set is a line; the least-norm solution is the point closest to the origin.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- 2D: x1 + 2*x2 = 3 ---
A_2d = np.array([[1., 2.]])
b_2d = np.array([3.])
t = np.linspace(-3, 3, 300)
# solution set: x1 = 3 - 2t, x2 = t
soln_x = 3 - 2*t
soln_y = t
x_star_2d = least_norm_solution(A_2d, b_2d)

axes[0].plot(soln_x, soln_y, 'b-', lw=2, label='Solution set $x_1+2x_2=3$')
axes[0].plot(*x_star_2d, 'ro', ms=10, label=f'$x^*$ = ({x_star_2d[0]:.2f}, {x_star_2d[1]:.2f})')
axes[0].plot(0, 0, 'k+', ms=10, label='Origin')
axes[0].plot([0, x_star_2d[0]], [0, x_star_2d[1]], 'r--', alpha=0.5)
axes[0].set_aspect('equal'); axes[0].grid(True); axes[0].legend()
axes[0].set_title('Solution set of $x_1 + 2x_2 = 3$'); axes[0].set_xlabel('$x_1$'); axes[0].set_ylabel('$x_2$')

# --- 3D: x1 + x2 + x3 = 2 ---
A_3d = np.array([[1., 1., 1.]])
b_3d = np.array([2.])
x_star_3d = least_norm_solution(A_3d, b_3d)
K_3d = null_space(A_3d)  # 2D kernel

# Sample points on the solution plane
alpha_vals = np.random.randn(400, 2)
pts = x_star_3d + (K_3d @ alpha_vals.T).T

ax = axes[1]
ax.remove(); ax = fig.add_subplot(1, 2, 2, projection='3d')
ax.scatter(pts[:,0], pts[:,1], pts[:,2], s=3, alpha=0.2, label='Solution set')
ax.scatter(*x_star_3d, color='red', s=100, zorder=5, label=f'$x^*$=({x_star_3d[0]:.1f},{x_star_3d[1]:.1f},{x_star_3d[2]:.1f})')
ax.plot([0, x_star_3d[0]], [0, x_star_3d[1]], [0, x_star_3d[2]], 'r--')
ax.set_title('Solution set of $x_1+x_2+x_3=2$ (a plane)')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()